# Week 9: Change Detection & Validation — ARIA v6.0## 第九週：變化偵測與驗證 — ARIA v6.0（經過驗證的稽核員）**Course**: 遙測與空間資訊之分析與應用 | Remote Sensing Analysis & Applications**Institution**: National Taiwan University (NTU)**Instructor**: Prof. Su Wen-Ray (蘇文瑄教授)**Case Study**: Matai'an Barrier Lake (Typhoon Colo)### Lab Rhythm / 實驗時間分配| Lab | Topic | Duration | 主題 ||-----|-------|----------|------|| **Lab 1** | Difference Mapping (NDVI/NDWI) | 35 min | 差異圖製作 || **Lab 2** | Accuracy Assessment & Validation | 50 min | 精度評估與驗證 |---### Context / 背景知識Pre-event baseline (Jun 2025) → Disaster onset (Aug 2025) → Post-event recovery (Oct 2025)**Study area (Matai'an)**: [121.28, 23.56, 121.52, 23.76]**Key indices**: NDVI (vegetation), NDWI (water)

## Lab 1: Difference Mapping — Which Index Reveals the Lake?### 實驗1：差異圖製作 — 哪個指標最清楚顯示水位變化？**Objective**: Compute spectral indices (NDVI, NDWI) for three temporal scenes, create difference maps, and explore threshold sensitivity.**步驟**:1. Load Sentinel-2 imagery from STAC2. Compute NDVI and NDWI for each scene3. Create difference maps: ΔIndex = Mid - Pre, Post - Mid4. Plot 2×2 panel showing change5. Sweep threshold and visualize detection area

In [ ]:
# [S1] Environment Setup# ──────────────────────────────────────────────────────────────────import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom datetime import datetimeimport warningswarnings.filterwarnings('ignore')# TODO: Import additional libraries (uncomment and complete)# from pystac_client import Client# from odc.stac import load as stac_load# from sklearn.metrics import confusion_matrix, classification_report# import json# import os# ConfigurationMATAIAN_BBOX = [121.28, 23.56, 121.52, 23.76]  # [W, S, E, N]
# ── Lake-focused AOI (NEW in W9) ──
# The barrier lake is ~1 km² in the upper-left of the 24×20 km scene.
# Using the full scene inflates OA with trivially-correct "stable" pixels.
# ── 堰塞湖只有 ~1 km²，用全場景會讓大量穩定像素灌水 OA。
LAKE_BBOX_LONLAT = [121.27, 23.68, 121.32, 23.72]   # ~5×4 km around lake

# ── SCL cloud masking (NEW in W9) ──
# SCL (Scene Classification Layer) classifies each pixel:
#   0=No data, 1=Saturated, 2=Dark, 3=Cloud shadow,
#   4=Vegetation, 5=Bare soil, 6=Water, 7=Unclassified,
#   8=Cloud (medium), 9=Cloud (high), 10=Thin cirrus, 11=Snow/ice
# We keep classes that represent clear-sky surface pixels.
# ── 保留代表晴空地表的類別。
SCL_CLEAR_CLASSES = [2, 4, 5, 6, 7, 11]

# TODO: Implement stream_scl() function
# def stream_scl(item, bbox=MATAIAN_BBOX):
#     """Stream SCL and return boolean clear-sky mask."""
#     signed = pc.sign(item)
#     scl = stackstac.stack(
#         [signed], assets=["SCL"], epsg=32651,
#         resolution=10, bounds_latlon=bbox, chunksize=2048
#     ).squeeze("time").squeeze("band")
#     scl_data = safe_compute(scl)
#     return np.isin(scl_data.values, SCL_CLEAR_CLASSES)
OUTPUT_DIR = "output"os.makedirs(OUTPUT_DIR, exist_ok=True)# Plot styleplt.style.use('seaborn-v0_8-darkgrid')sns.set_palette("husl")print("✓ Environment setup complete")print(f"  Output directory: {OUTPUT_DIR}")print(f"  Study area BBOX: {MATAIAN_BBOX}")

In [ ]:
# [S2] Search and Load Three-Act Scenes# ──────────────────────────────────────────────────────────────────# Search STAC for Pre, Mid, Post scenes of Matai'an barrier lake# Scene dates (three acts of the disaster narrative)SCENE_DATES = {    'Pre': '2025-06-01',    # Baseline (before typhoon)    'Mid': '2025-08-15',    # Lake present at peak    'Post': '2025-10-10'    # Lake drained / recovered}# TODO: Implement robust STAC search# Hints:#   1. Create STAC client (e.g., Sentinel-2 L2A collections)#   2. Search for each date within MATAIAN_BBOX#   3. Use robust_search() helper to handle failed queries#   4. Store results in dict: scenes = {'Pre': {...}, 'Mid': {...}, 'Post': {...}}# Example stub (replace with working code):# def robust_search(client, bbox, start_date, end_date, collection='sentinel-2-l2a'):#     try:#         results = client.search(#             collections=[collection],#             bbox=bbox,#             datetime=f'{start_date}T00:00:00Z/{end_date}T23:59:59Z',#             max_items=1#         )#         return results.item_collection(10).to_dict()#     except Exception as e:#         print(f"Search failed: {e}")#         return None# TODO: Complete the search logicscenes = {}# for act, date in SCENE_DATES.items():#     print(f"Searching {act} ({date})...")#     # Insert robust_search call here#     passprint("✓ Scene search complete")print(f"  Scenes found: {list(scenes.keys())}")

In [ ]:
# [S3] Load and Composite Multi-Spectral Cubes# ──────────────────────────────────────────────────────────────────# TODO: Implement stream_cube() to load Sentinel-2 bands and create stretched composite# Reference bands: B02 (Blue), B03 (Green), B04 (Red), B08 (NIR), B11 (SWIR)BANDS = ['B02', 'B03', 'B04', 'B08', 'B11']BAND_NAMES = {    'B02': 'Blue',    'B03': 'Green',    'B04': 'Red',    'B08': 'NIR',    'B11': 'SWIR'}# TODO: Define stream_cube() function# def stream_cube(scene_item, bands):#     '''Load bands from scene item and stack into cube (H, W, C)'''#     cube = np.stack([load_band(scene_item, band) for band in bands], axis=-1)#     return cube / 10000.0  # Scale to [0, 1]# TODO: Define composite_stretched() function# def composite_stretched(cube, r_idx=2, g_idx=1, b_idx=0, q=(2, 98)):#     '''Create RGB composite with percentile-based contrast stretch'''#     rgb = cube[:, :, [r_idx, g_idx, b_idx]].copy()#     for i in range(3):#         p_lo, p_hi = np.percentile(rgb[:,:,i], q)#         rgb[:,:,i] = np.clip((rgb[:,:,i] - p_lo) / (p_hi - p_lo), 0, 1)#     return rgb# TODO: Load and composite each scene# cubes = {}# composites = {}# for act in ['Pre', 'Mid', 'Post']:#     print(f"Loading {act}...")#     cubes[act] = stream_cube(scenes[act], BANDS)#     composites[act] = composite_stretched(cubes[act])
# ── Cloud masking (CRITICAL — without this, ΔNDWI shows "phantom water") ──
# TODO: Stream SCL for each scene and create masks
# clear_pre  = stream_scl(pre_item)
# clear_mid  = stream_scl(mid_item)
# clear_post = stream_scl(post_item)

# ── Per-scene valid masks (for Three-Act visualization) ──
# Each scene uses its OWN mask so students can see Mid has more clouds.
# TODO:
# valid_pre  = np.isfinite(nir_pre)  & clear_pre
# valid_mid  = np.isfinite(nir_mid)  & clear_mid
# valid_post = np.isfinite(nir_post) & clear_post

# ── Intersection mask (for difference maps where BOTH dates must be valid) ──
# TODO:
# valid = (valid_raw & clear_pre & clear_mid & clear_post)
print("✓ Cubes loaded and composited")print(f"  Cube shape: (H, W, C) = {list(cubes.values())[0].shape if cubes else 'N/A'}")

### Computing Spectral Indices: NDVI & NDWI### 計算光譜指標：NDVI & NDWI**NDVI** = (NIR - Red) / (NIR + Red)**NDWI** = (Green - NIR) / (Green + NIR)─Why two indices?- **NDVI**: Sensitive to vegetation density, but also responds to water- **NDWI**: Maximally sensitive to open water and wetlands─為何使用兩種指標？- **NDVI**：對植被濃度敏感，但也對水體有反應- **NDWI**：對開放水體和濕地最敏感

In [ ]:
# [S4] Compute NDVI and NDWI for All Three Scenes# ──────────────────────────────────────────────────────────────────# TODO: Implement ndvi() and ndwi() functions# def ndvi(cube):#     '''Compute NDVI from cube (B04=Red at idx 2, B08=NIR at idx 3)'''#     red = cube[:, :, 2]#     nir = cube[:, :, 3]#     ndvi = (nir - red) / (nir + red + 1e-8)#     return ndvi# def ndwi(cube):#     '''Compute NDWI from cube (B03=Green at idx 1, B08=NIR at idx 3)'''#     green = cube[:, :, 1]#     nir = cube[:, :, 3]#     ndwi = (green - nir) / (green + nir + 1e-8)#     return ndwi# TODO: Compute indices for all three scenes# indices = {}# for act in ['Pre', 'Mid', 'Post']:#     indices[act] = {#         'NDVI': ndvi(cubes[act]),#         'NDWI': ndwi(cubes[act])#     }# Print summary statistics# for act in ['Pre', 'Mid', 'Post']:#     print(f"{act} scene:")#     for idx_name in ['NDVI', 'NDWI']:#         idx = indices[act][idx_name]#         print(f"  {idx_name}: μ={idx.mean():.3f}, σ={idx.std():.3f}, "#               f"[{idx.min():.3f}, {idx.max():.3f}]")print("✓ NDVI and NDWI computed")print("  (Uncomment the TODO blocks to visualize)")

In [ ]:
# [S5] Create Difference Maps: ΔIndex and Visualize 2×2 Panel# ──────────────────────────────────────────────────────────────────# TODO: Compute difference maps (Mid - Pre, Post - Mid)# d_ndvi = {}# d_ndwi = {}## # TODO: Compute Mid - Pre# d_ndvi['Mid-Pre'] = indices['Mid']['NDVI'] - indices['Pre']['NDVI']# d_ndwi['Mid-Pre'] = indices['Mid']['NDWI'] - indices['Pre']['NDWI']## # TODO: Compute Post - Mid# d_ndvi['Post-Mid'] = indices['Post']['NDVI'] - indices['Mid']['NDVI']# d_ndwi['Post-Mid'] = indices['Post']['NDWI'] - indices['Mid']['NDWI']# TODO: Create 2×2 subplot showing both indices and both transitions# fig, axes = plt.subplots(2, 2, figsize=(12, 10))## im0 = axes[0, 0].imshow(d_ndvi['Mid-Pre'], cmap='RdBu_r', vmin=-0.3, vmax=0.3)# axes[0, 0].set_title('ΔNDVI (Mid - Pre)')# plt.colorbar(im0, ax=axes[0, 0])## im1 = axes[0, 1].imshow(d_ndwi['Mid-Pre'], cmap='RdBu_r', vmin=-0.3, vmax=0.3)# axes[0, 1].set_title('ΔNDWI (Mid - Pre)')# plt.colorbar(im1, ax=axes[0, 1])## im2 = axes[1, 0].imshow(d_ndvi['Post-Mid'], cmap='RdBu_r', vmin=-0.3, vmax=0.3)# axes[1, 0].set_title('ΔNDVI (Post - Mid)')# plt.colorbar(im2, ax=axes[1, 0])## im3 = axes[1, 1].imshow(d_ndwi['Post-Mid'], cmap='RdBu_r', vmin=-0.3, vmax=0.3)# axes[1, 1].set_title('ΔNDWI (Post - Mid)')# plt.colorbar(im3, ax=axes[1, 1])## for ax in axes.flat:#     ax.set_xlabel('X (pixels)')#     ax.set_ylabel('Y (pixels)')## plt.tight_layout()# plt.savefig(f'{OUTPUT_DIR}/W9_L1_difference_maps.png', dpi=150, bbox_inches='tight')# plt.show()print("✓ Difference maps created (uncomment code to visualize)")print("  Output: W9_L1_difference_maps.png")

### Error Case: What Happens WITHOUT Cloud Masking?
### 錯誤案例：不做雲遮罩會怎樣？

Before computing difference maps, compare results WITH and WITHOUT cloud masking.
This demonstrates why SCL masking is **mandatory** for change detection.

**Task**: Create a 3-panel comparison:
1. ❌ ΔNDWI without cloud mask (`valid_raw`) — "phantom water" everywhere
2. ✅ ΔNDWI with cloud mask (`valid`) — only real water changes
3. 🔍 ΔNDWI zoomed to lake AOI (`LAKE_BBOX_LONLAT`) — focused view

── 計算差異圖之前，比較有無雲遮罩的結果。
── 這展示了為什麼 SCL 遮罩在變遷偵測中是**必要的**。


### Discussion: Which Index Shows the Barrier Lake Best?### 討論：哪個指標最清楚顯示障礙湖？**Observations** (from the 2×2 panel):1. **ΔNDVI (Mid - Pre)**: Does vegetation decrease at the lake site? Why might it stay positive?   - NDVI = (NIR - Red)/(NIR + Red) → water has low NDVI, but may not be negative2. **ΔNDWI (Mid - Pre)**: Is this more clearly showing water presence?   - NDWI = (Green - NIR)/(Green + NIR) → water has high NDWI3. **Recovery phase (Post - Mid)**: Do both indices return to baseline?─**Student reflection questions**:- Which index is more robust to atmospheric noise and vegetation near the shoreline?- Why might NDWI be designed specifically for water detection?- What's the trade-off between specificity (only water) and sensitivity (catching all changes)?

In [ ]:
# [S6] Threshold Sensitivity Demo# ──────────────────────────────────────────────────────────────────# TODO: Sweep thresholds across NDWI and measure detected area# Function to count pixels above thresholddef detect_water(index_map, threshold):    '''Count pixels where index > threshold (assumed water signal)'''    return (index_map > threshold).sum()# TODO: Sweep thresholds and compute detection area# thresholds = [-0.10, -0.20, -0.30, -0.40, -0.50]  # TODO: Demo thresholds (update comment)# detected_areas = {}## for act_pair in ['Mid-Pre', 'Post-Mid']:#     areas = []#     for t in thresholds:#         area = detect_water(d_ndwi[act_pair], t)#         areas.append(area)#     detected_areas[act_pair] = np.array(areas)# TODO: Plot detection area vs threshold# fig, ax = plt.subplots(figsize=(10, 6))# ax.plot(thresholds, detected_areas['Mid-Pre'], 'o-', label='Mid - Pre (Lake appearing)', linewidth=2)# ax.plot(thresholds, detected_areas['Post-Mid'], 's-', label='Post - Mid (Lake draining)', linewidth=2)# ax.axvline(0.1, color='red', linestyle='--', label='Threshold = 0.1', alpha=0.7)# ax.set_xlabel('NDWI Threshold', fontsize=12)# ax.set_ylabel('Detected Area (pixels)', fontsize=12)# ax.set_title('Threshold Sensitivity: How Much Water Detected?', fontsize=14)# ax.legend(fontsize=11)# ax.grid(True, alpha=0.3)# plt.tight_layout()# plt.savefig(f'{OUTPUT_DIR}/W9_L1_threshold_sensitivity.png', dpi=150, bbox_inches='tight')# plt.show()print("✓ Threshold sensitivity analysis prepared (uncomment code to run)")print("  Output: W9_L1_threshold_sensitivity.png")

### Discussion: Threshold is a DECISION, Not a Formula### 討論：臨界值是「決策」，不是「公式」**Key insight**: No single "correct" threshold exists. The choice depends on your **use case**:| Use Case | Threshold | Rationale | 優先考量 ||----------|-----------|-----------|---------|| **Disaster Alert** | Low (0.05) | Catch early signals, high false positive OK | 敏感度 || **Insurance Payout** | Medium (0.10) | Balance detection & accuracy | 平衡 || **Scientific Archive** | High (0.15) | Minimize false positives | 特異性 |─**Question for reflection**:- In the Oct 2025 post-event phase, should ARIA v6.0 use a low or high threshold?- What are the consequences of each choice for emergency responders?

## Lab 2: Accuracy Assessment & Validation### 實驗2：精度評估與驗證**Objective**: Load validation ground truth points, sample detection masks, compute confusion matrix, and calculate accuracy metrics.**Framework**: ARIA v6.0 (Auditor + Rater + Indicator + Advisor)**步驟**:1. Load validation_points.geojson (labeled samples: Water / No Water)2. Build detection masks using threshold from Lab 13. Sample each mask at validation point locations4. Compute confusion matrix5. Calculate Producer's Accuracy, User's Accuracy, Overall Accuracy, Kappa6. Visualize confusion matrix heatmap7. Find optimal threshold via F1 score8. Create confidence map (3-zone)9. Generate validation report

### How to Build Real Ground Truth / 如何建立真實地面驗證資料

**This lab uses 60 field-corrected validation points provided by the instructor.**  
In a real remote sensing project, you must collect **independent** ground truth. Here are the main approaches, ordered from most to least reliable:

| Method | Description | Cost | When to Use |
|--------|-------------|------|-------------|
| **Field survey + GPS** | Walk to locations, record water/no-water with GPS coordinates | High | Gold standard for small areas |
| **UAV / Drone imagery** | High-resolution aerial photos (~5 cm/px) | Medium | Medium areas, post-disaster access |
| **Google Earth Pro time-series** | Compare VHR imagery before/after event | Free | Historical events, desktop validation |
| **NCDR disaster reports** | 國家災害防救科技中心 official damage assessments | Free | Taiwan-specific disasters |
| **Copernicus EMS** | EU emergency mapping service, rapid activation maps | Free | Global disasters with EMS activation |
| **News media + geotagged photos** | Cross-reference reported flood areas | Free | Quick initial reference |

**Key principles:**
1. **Independence**: Validation data must come from a source *other than* the satellite imagery you're analyzing
2. **Stratified sampling**: Distribute points across all zones (flooded, unflooded, boundary)
3. **Sufficient sample size**: At least 30–50 points; 100+ for publishable accuracy
4. **Temporal match**: Ground truth date should be close to satellite acquisition date

> **作業提示**: Homework Task 3 (Optional) 鼓勵你用 Google Earth Pro 自行標註 20+ 個驗證點。  
> 這才是遙測分析的正確做法——自己建立可信的驗證資料集。

In [ ]:
# [S7] Build Detection Masks (Index-Based Logic)# ──────────────────────────────────────────────────────────────────# TODO: Threshold NDWI to create binary water masks# Define threshold for water detectionTHRESHOLD_NDWI = 0.10  # Tunable (reference from Lab 1 threshold sweep)# TODO: Create binary masks for Mid and Post scenes# masks = {}# masks['Mid'] = (indices['Mid']['NDWI'] > THRESHOLD_NDWI).astype(int)  # 1=Water, 0=No Water# masks['Post'] = (indices['Post']['NDWI'] > THRESHOLD_NDWI).astype(int)# Visualize masks# fig, axes = plt.subplots(1, 2, figsize=(12, 5))## im0 = axes[0].imshow(masks['Mid'], cmap='RdYlBu', vmin=0, vmax=1)# axes[0].set_title(f'Water Mask (Mid, threshold={THRESHOLD_NDWI})')# plt.colorbar(im0, ax=axes[0], label='Water')## im1 = axes[1].imshow(masks['Post'], cmap='RdYlBu', vmin=0, vmax=1)# axes[1].set_title(f'Water Mask (Post, threshold={THRESHOLD_NDWI})')# plt.colorbar(im1, ax=axes[1], label='Water')## for ax in axes:#     ax.set_xlabel('X (pixels)')#     ax.set_ylabel('Y (pixels)')## plt.tight_layout()# plt.savefig(f'{OUTPUT_DIR}/W9_L2_masks.png', dpi=150, bbox_inches='tight')# plt.show()print(f"✓ Detection masks created (threshold={THRESHOLD_NDWI})")print("  Mask shape:", list(masks.values())[0].shape if masks else "N/A")

In [ ]:
# [S8] Load Validation Points and Sample Masks
# ──────────────────────────────────────────────────────────────────
#
# The teacher provides data/validation_points.geojson with 60 points:
#   - 15 lake, 15 landslide, 30 stable
#   - Coordinates corrected by instructor using VHR imagery + NCDR reports
#
# TODO: Load the geojson and extract lon/lat from geometry
# ──────────────────────────────────────────────────────────────────

import json as _json

# Load geojson
with open("data/validation_points.geojson") as _f:
    _gj = _json.load(_f)

val_rows = []
for feat in _gj["features"]:
    lon, lat = feat["geometry"]["coordinates"]
    val_rows.append({"lon": lon, "lat": lat,
                     "truth": feat["properties"]["truth"],
                     "source": feat["properties"]["source"]})

validation_points = pd.DataFrame(val_rows)

print(f"✅ Loaded {len(validation_points)} validation points")
print(f"  Source: {validation_points['source'].unique()}")
print(f"  Ground truth distribution:")
print(validation_points['truth'].value_counts().to_string())

# TODO: Convert geographic coordinates to pixel indices
# def geo_to_pixel(lon, lat, geotransform):
#     col = int((lon - geotransform[0]) / geotransform[1])
#     row = int((geotransform[3] - lat) / -geotransform[5])
#     return row, col


In [ ]:
# [S9] Compute Confusion Matrix
# ──────────────────────────────────────────────────────────────────
# TODO: Sample your detection mask at ground truth locations.
#    For now, a demo confusion matrix is shown below.
# ──────────────────────────────────────────────────────────────────

# TODO: In real workflow, sample predicted mask at validation points:
# validation_points['predicted'] = [mask[geo_to_pixel(lon, lat, gt)] ...]
# cm = confusion_matrix(ground_truth, predicted, labels=[0, 1])

# ── Synthetic confusion matrix for demo ──
predicted = np.random.randint(0, 2, len(validation_points))
ground_truth = validation_points['ground_truth'].values

cm = np.array([[12, 3],   # TN=12, FP=3
               [2, 13]])   # FN=2, TP=13

print("✓ Confusion matrix computed (demo values)")
print("\nConfusion Matrix:")
print(cm)
print("\nLabels: [0=No Water, 1=Water]")
print(f"  TN (True Negatives): {cm[0,0]}")
print(f"  FP (False Positives): {cm[0,1]}")
print(f"  FN (False Negatives): {cm[1,0]}")
print(f"  TP (True Positives): {cm[1,1]}")


In [ ]:
# [S10] Compute Accuracy Metrics# ──────────────────────────────────────────────────────────────────# TODO: Calculate Producer's Accuracy, User's Accuracy, Overall Accuracy, Kappa# Extract confusion matrix elementstn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]# TODO: Producer's Accuracy (Sensitivity, Recall, True Positive Rate)producer_accuracy = tp / (tp + fn)# TODO: User's Accuracy (Precision, Positive Predictive Value)user_accuracy = tp / (tp + fp)# TODO: Overall Accuracyoverall_accuracy = (tp + tn) / (tp + tn + fp + fn)# TODO: Cohen's Kappa (accounts for chance agreement)# po = overall_accuracy (observed agreement)# pe = expected agreement by chancepe = ((tp + fn) * (tp + fp) + (tn + fp) * (tn + fn)) / (tp + tn + fp + fn)**2kappa = (overall_accuracy - pe) / (1 - pe)print("✓ Accuracy metrics computed\n")print("=" * 50)print("ARIA v6.0 VALIDATION METRICS")print("=" * 50)print(f"Producer's Accuracy (Sensitivity): {producer_accuracy:.3f} ({producer_accuracy*100:.1f}%)")print(f"User's Accuracy (Precision):       {user_accuracy:.3f} ({user_accuracy*100:.1f}%)")print(f"Overall Accuracy:                  {overall_accuracy:.3f} ({overall_accuracy*100:.1f}%)")print(f"Cohen's Kappa:                     {kappa:.3f}")print("=" * 50)# Store metrics for later usemetrics = {    'producer_accuracy': producer_accuracy,    'user_accuracy': user_accuracy,    'overall_accuracy': overall_accuracy,    'kappa': kappa,    'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn}

In [ ]:
# [S11] Plot Confusion Matrix Heatmap# ──────────────────────────────────────────────────────────────────# TODO: Create confusion matrix heatmap with annotationsfig, ax = plt.subplots(figsize=(8, 7))# Create heatmapim = ax.imshow(cm, cmap='Blues', aspect='auto')# Add text annotationsfor i in range(2):    for j in range(2):        text = ax.text(j, i, cm[i, j],                      ha="center", va="center", color="black", fontsize=16, fontweight='bold')# Labelsax.set_xticks([0, 1])ax.set_yticks([0, 1])ax.set_xticklabels(['No Water', 'Water'], fontsize=12)ax.set_yticklabels(['No Water', 'Water'], fontsize=12)ax.set_xlabel('Predicted Class', fontsize=12, fontweight='bold')ax.set_ylabel('Ground Truth Class', fontsize=12, fontweight='bold')ax.set_title('Confusion Matrix: Water Detection\nARIA v6.0 Validation', fontsize=14, fontweight='bold')# Colorbarcbar = plt.colorbar(im, ax=ax)cbar.set_label('Count', fontsize=11)plt.tight_layout()plt.savefig(f'{OUTPUT_DIR}/W9_L2_confusion_matrix.png', dpi=150, bbox_inches='tight')plt.show()print("✓ Confusion matrix heatmap saved")print(f"  Output: W9_L2_confusion_matrix.png")

### Discussion: What Do These Accuracy Numbers Mean for Disaster Response?### 討論：這些精度數字對災難應對意味著什麼？Consider ARIA's role in Typhoon Colo response (Aug 2025):1. **Producer's Accuracy (Sensitivity) = 87%**   - Out of all actual water pixels, we correctly identified 87%   - Missing 13% of the true lake area   - Implication: Evacuation zone might miss some flooded areas2. **User's Accuracy (Precision) = 81%**   - Out of all pixels we called "water", 81% actually are water   - 19% false alarms (crying wolf)   - Implication: Emergency responders waste resources on false alarms3. **Overall Accuracy = 83%**   - 83% of all predictions correct   - Looks good, but hides class imbalance if water is rare─**Critical question**: In a disaster setting, which error is worse?- **False Negative** (missing flooded area) → people in danger- **False Positive** (false alarm) → wasted resources, lost credibilityFor early warning systems, we often prefer **high sensitivity** (low false negatives) over precision.

### Discussion: Why Is Producer's Accuracy More Important Than Overall Accuracy?### 討論：為何生產者精度比整體精度更重要？**Scenario**: Matai'an barrier lake covers ~10% of the study area. Rest is land/forest.If model predicts: "Everything is NOT water"- TN = 90%, FP = 0%, TP = 0%, FN = 10%- **Overall Accuracy** = 90% ✓ (looks amazing!)- **Producer's Accuracy (Sensitivity)** = 0% ✗ (catastrophic: missed all the water!)This is called the **accuracy paradox** or **class imbalance problem**.─**For disaster detection**:- We care most about **not missing disasters** → Prioritize **Sensitivity / Producer's Accuracy**- Trade-off: Accept some false positives (precision tradeoff)- This is why early warning systems often have lower precision but higher sensitivity**ARIA v6.0 design principle**:> "Better to alert 10 safe neighborhoods than miss 1 at-risk community."

In [ ]:
# [S12] F1 Score vs Threshold: Finding Optimal Detection Threshold
# ──────────────────────────────────────────────────────────────────
# TODO: Sweep thresholds over the loaded validation points.
#    For now, a demo F1 curve is shown below.
# ──────────────────────────────────────────────────────────────────
#
# F1 = 2 * (precision * recall) / (precision + recall)
# The F1 score is the harmonic mean of precision and recall.
#
# TODO: For each threshold, build mask → sample at validation pts → compute F1
# thresholds = np.linspace(-0.50, 0.00, 50)
# for t in thresholds:
#     mask = delta_ndvi < t
#     predicted = sample_mask_at_points(mask, validation_points)
#     f1 = f1_score(ground_truth, predicted)

# ── Synthetic F1 curve for demo ──
thresholds_f1 = np.linspace(0.05, 0.25, 30)
f1_scores = 0.85 + 0.10 * np.sin((thresholds_f1 - 0.10) * 30)  # Synthetic curve
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds_f1[optimal_idx]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresholds_f1, f1_scores, 'o-', linewidth=2, markersize=6, label='F1 Score')
ax.axvline(optimal_threshold, color='red', linestyle='--', linewidth=2,
           label=f'Optimal Threshold = {optimal_threshold:.3f}')
ax.scatter([optimal_threshold], [f1_scores[optimal_idx]], color='red', s=200, zorder=5, marker='*')
ax.set_xlabel('NDWI Threshold', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Optimal Threshold Selection via F1 Score (Demo)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.7, 0.95])
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/W9_L2_f1_threshold.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ F1 vs threshold analysis complete (demo)")
print(f"  Optimal threshold: {optimal_threshold:.3f}")
print(f"  Optimal F1 score: {f1_scores[optimal_idx]:.3f}")


In [ ]:
# [S13] Build 3-Zone Confidence Map: High / Low / None# ──────────────────────────────────────────────────────────────────# TODO: Create confidence zones using thresholds# Define confidence thresholdsTHRESHOLD_LOW = 0.05     # Possible water (low confidence)THRESHOLD_HIGH = 0.15    # Likely water (high confidence)# Create confidence mask using Mid-scene NDWIndwi_mid = indices['Mid']['NDWI']confidence_map = np.zeros_like(ndwi_mid, dtype=int)confidence_map[ndwi_mid > THRESHOLD_HIGH] = 2  # High confidence (water)confidence_map[(ndwi_mid > THRESHOLD_LOW) & (ndwi_mid <= THRESHOLD_HIGH)] = 1  # Low confidence# confidence_map[ndwi_mid <= THRESHOLD_LOW] = 0  # No signal (default)# Visualize 3-zone mapfig, ax = plt.subplots(figsize=(10, 8))im = ax.imshow(confidence_map, cmap='RdYlGn_r', vmin=0, vmax=2)ax.set_title('ARIA v6.0 Confidence Map: Water Detection Uncertainty', fontsize=14, fontweight='bold')ax.set_xlabel('X (pixels)')ax.set_ylabel('Y (pixels)')# Colorbar with custom labelscbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2])cbar.ax.set_yticklabels(['No Signal', 'Low Confidence', 'High Confidence'])cbar.set_label('Confidence Level')plt.tight_layout()plt.savefig(f'{OUTPUT_DIR}/W9_L2_confidence_map.png', dpi=150, bbox_inches='tight')plt.show()print("✓ 3-zone confidence map created")print(f"  High confidence pixels: {(confidence_map == 2).sum()}")print(f"  Low confidence pixels: {(confidence_map == 1).sum()}")print(f"  No signal pixels: {(confidence_map == 0).sum()}")

In [ ]:
# [S14] ARIA v6.0 Validated Disaster Report# ──────────────────────────────────────────────────────────────────report = f"""╔════════════════════════════════════════════════════════════════════════════════╗║                     ARIA v6.0 VALIDATED DISASTER REPORT                        ║║              Remote Sensing Analysis & Validation Authority (ARIA)             ║╚════════════════════════════════════════════════════════════════════════════════╝CASE STUDY: Matai'an Barrier Lake (Typhoon Colo Response)Institution: National Taiwan University (NTU)Analyst: Remote Sensing LabReport Date: 2025-10-12────────────────────────────────────────────────────────────────────────────────EXECUTIVE SUMMARY─────────────────A temporary barrier lake formed in Matai'an valley during Typhoon Colo (Aug 2025).Using Sentinel-2 change detection, we mapped water presence and validated againstground-truth validation points. NDWI proved more sensitive than NDVI for water.STUDY AREA (Bounding Box)──────────────────────────West:  121.28° EEast:  121.52° ESouth: 23.56° NNorth: 23.76° NArea:  ~360 km²TEMPORAL ANALYSIS─────────────────Pre-event baseline:   2025-06-01 (before typhoon)Disaster onset:       2025-08-15 (lake at maximum extent)Post-event recovery:  2025-10-10 (lake drained)SPECTRAL INDICES USED─────────────────────NDVI = (NIR - Red) / (NIR + Red)  → Sensitivity: Vegetation density, weak water signal  → ΔNDVIpeak = {indices['Mid']['NDVI'].mean() - indices['Pre']['NDVI'].mean():.4f}NDWI = (Green - NIR) / (Green + NIR)  → Sensitivity: Maximum water detection  → ΔNDWIpeak = {indices['Mid']['NDWI'].mean() - indices['Pre']['NDWI'].mean():.4f}VALIDATION RESULTS──────────────────Validation Points Analyzed:    {len(validation_points)}Ground Truth Classes:          Water={validation_points['ground_truth'].sum()}, No-Water={len(validation_points) - validation_points['ground_truth'].sum()}Confusion Matrix (Water Detection at Mid-event):  True Negatives:     {metrics['tn']:>3.0f}  False Positives:    {metrics['fp']:>3.0f}  False Negatives:    {metrics['fn']:>3.0f}  True Positives:     {metrics['tp']:>3.0f}ACCURACY METRICS────────────────Producer's Accuracy (Sensitivity / Recall):  {metrics['producer_accuracy']:.3f} ({metrics['producer_accuracy']*100:.1f}%)  → {int(metrics['fn'])} false negatives (missed water)User's Accuracy (Precision):  {metrics['user_accuracy']:.3f} ({metrics['user_accuracy']*100:.1f}%)  → {int(metrics['fp'])} false positives (false alarms)Overall Accuracy:  {metrics['overall_accuracy']:.3f} ({metrics['overall_accuracy']*100:.1f}%)Cohen's Kappa (Agreement beyond chance):  {metrics['kappa']:.3f}  → Interpretation: Substantial agreement (0.61-0.80 = Substantial; >0.80 = Almost Perfect)DETECTION THRESHOLD────────────────────Selected NDWI Threshold:  {THRESHOLD_NDWI:.3f}Rationale:               Balance between sensitivity and precisionOptimal Threshold (F1):  {optimal_threshold:.3f}CONFIDENCE LEVELS──────────────────High Confidence Zone (NDWI > 0.15):  → Most likely water; high confidence for evacuation planningLow Confidence Zone (0.05 < NDWI ≤ 0.15):  → Possible water; requires field verification or higher-res imageryNo Signal Zone (NDWI ≤ 0.05):  → No detectable water; safe for reconstruction activitiesRECOMMENDATIONS FOR DISASTER RESPONSE──────────────────────────────────────1. EVACUATION PRIORITY: High-confidence zones should be prioritized for evacuation2. RESOURCE ALLOCATION: Low-confidence zones warrant additional ground surveys3. RECOVERY TIMELINE: October imagery confirms lake drainage; safe to begin recovery ops4. THRESHOLD TUNING: Producer's Accuracy is critical; prioritize sensitivity over precisionKEY FINDINGS────────────✓ NDWI is more reliable than NDVI for barrier lake detection✓ Sentinel-2 adequate for large-scale (>100 km²) water mapping✓ Temporal change (Mid - Pre) more robust than absolute thresholds✓ Ground validation essential for disaster response decisionsLIMITATIONS & CAVEATS─────────────────────• Cloud cover reduces valid pixels (typical for tropical typhoon season)• Shoreline pixels mixed (land-water transitions) introduce uncertainty• Validation points limited; larger sample recommended for operational use• Resolution (~10m) may miss small isolated ponds────────────────────────────────────────────────────────────────────────────────ARIA v6.0 ASSESSMENT: ✓ VALIDATED & READY FOR OPERATIONAL USEAuditor (Confusion Matrix):     ✓ ComputedRater (Accuracy Metrics):       ✓ CalculatedIndicator (Confidence Zones):   ✓ MappedAdvisor (Recommendations):      ✓ ProvidedReport Status: FINALQuality Level: Operational (Ready for Emergency Response)────────────────────────────────────────────────────────────────────────────────Generated by ARIA v6.0 at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"""print(report)# Save reportwith open(f'{OUTPUT_DIR}/ARIA_v6_0_Disaster_Report.txt', 'w', encoding='utf-8') as f:    f.write(report)print("\n✓ Disaster report saved")print(f"  Output: ARIA_v6_0_Disaster_Report.txt")

In [ ]:
# [S15] AI Advisor: Prompt Template for Disaster Response Questions# ──────────────────────────────────────────────────────────────────advisor_prompt_template = """ARIA v6.0 AI ADVISOR - Ask Me About Your Disaster Response Questions═══════════════════════════════════════════════════════════════════════You are analyzing Sentinel-2 remote sensing data for Typhoon Colo impact assessment.Use the validation metrics and change maps above to answer operational questions.CONTEXT PROVIDED:  • Confusion matrix with TP/TN/FP/FN  • Producer's & User's Accuracy scores  • 3-zone confidence map (High/Low/No Signal)  • Pre/Mid/Post temporal scenes  • NDVI & NDWI difference mapsSAMPLE QUESTIONS YOU CAN ANSWER:1. "What's the maximum lake extent in the Mid-event scene?"   → Use high-confidence zone from confidence_map; estimate area in km²2. "How accurate is our water detection for evacuation planning?"   → Reference Producer's Accuracy; explain false negatives risk3. "Why might we have false positives along the lakeshore?"   → Vegetation/water mixing; spectral confusion at boundaries4. "Should we trust the Post-event recovery map?"   → Compare Post and Mid NDWI; check confidence zones5. "How would different thresholds affect our evacuation zone?"   → Reference threshold sensitivity plot; trade sensitivity vs precisionYOUR RESPONSE SHOULD:  ✓ Cite specific metrics from the validation report  ✓ Explain the uncertainty and confidence limits  ✓ Recommend actionable next steps  ✗ Avoid over-interpreting noisy pixels  ✗ Don't claim precision beyond 10m resolution────────────────────────────────────────────────────────────────────────────────To use this prompt with an LLM (e.g., ChatGPT):1. Paste this entire block + the ARIA report above2. Add your question about the disaster response3. Ask the AI to respond using only the provided dataExample user query:  "Based on this analysis, which zones should we evacuate first?"Example AI response:  "Based on the high-confidence water detection (NDWI > 0.15), zone XYZ should be   prioritized. However, note that our Producer's Accuracy is 87%, so we may be   missing ~13% of the actual water. Recommend ground surveys in low-confidence zones."────────────────────────────────────────────────────────────────────────────────"""print(advisor_prompt_template)with open(f'{OUTPUT_DIR}/AI_Advisor_Prompt_Template.txt', 'w', encoding='utf-8') as f:    f.write(advisor_prompt_template)print("✓ AI Advisor prompt template saved")print(f"  Output: AI_Advisor_Prompt_Template.txt")

## Wrap-Up Checklist / 實驗完成清單After completing both labs, verify you have:### Lab 1 Deliverables- [ ] NDVI and NDWI computed for all three scenes (Pre, Mid, Post)- [ ] 2×2 difference map panel showing ΔIndex changes- [ ] Threshold sensitivity curve (detected area vs NDWI threshold)- [ ] Discussion written: Which index best shows the barrier lake?### Lab 2 Deliverables- [ ] Water detection masks created (using optimal threshold)- [ ] Validation points sampled at mask locations- [ ] Confusion matrix computed- [ ] Accuracy metrics calculated: Producer's, User's, Overall, Kappa- [ ] Confusion matrix heatmap plotted- [ ] F1 vs threshold curve (optimal threshold found)- [ ] 3-zone confidence map created (High/Low/None)- [ ] ARIA v6.0 disaster report generated- [ ] Discussion written: Why is Producer's Accuracy critical for disaster response?### Files in Output Directory- [ ] `W9_L1_difference_maps.png` — 2×2 NDVI/NDWI panel- [ ] `W9_L1_threshold_sensitivity.png` — Threshold sweep curve- [ ] `W9_L2_masks.png` — Binary water masks- [ ] `W9_L2_confusion_matrix.png` — Heatmap- [ ] `W9_L2_f1_threshold.png` — Optimal threshold- [ ] `W9_L2_confidence_map.png` — 3-zone map- [ ] `ARIA_v6_0_Disaster_Report.txt` — Validation report- [ ] `AI_Advisor_Prompt_Template.txt` — LLM prompt### Reflection Questions (for course forum)1. What would change if we used a **lower** NDWI threshold (0.05 vs 0.15)?2. Why is **temporal change** (Mid - Pre) more robust than absolute NDWI values?3. If you were in disaster operations, would you prioritize **sensitivity** or **precision**?4. How would **cloud cover** affect the validation metrics?5. What role does **ground truth validation** play in building trust in automated systems?

---

## Before You Submit: Verify Your Work / 繳交前：驗證你的成果

> **⚠️ An unverified map is more dangerous than no map at all.**

Your notebook may run without errors but still produce **wrong results**. Before submitting:

| Check | Question to Ask Yourself |
|-------|--------------------------|
| **Metrics** | Does OA = 99.9%? That's almost certainly a bug. Does Kappa = 0? Your threshold may be wrong. |
| **Maps** | Does ΔNDWI show change at the lake site? Or random noise everywhere? |
| **Threshold** | Can you explain WHY you chose this threshold? |
| **Confusion Matrix** | Are TP/FP/TN/FN consistent with what you see in the map? |
| **Captain's Log** | Did you reflect on what the numbers mean, or just copy-paste? |

**跑出來 ≠ 跑對了。不懂可以問，但不要敷衍交差。**

If something looks wrong, ask on NTUCool or during office hours — that's what they're for.


## Environment Setup: .env Template### 環境設置：.env 模板If using STAC Client API keys or other credentials, create a `.env` file:```bash# .env template (NEVER commit this file to Git!)# STAC Planetary ComputerSTAC_ENDPOINT="https://planetarycomputer.microsoft.com/api/stac/v1"# Copernicus DataSpace (Sentinel-2 access)COPERNICUS_USERNAME="your_username"COPERNICUS_PASSWORD="your_password"# Optional: Google Earth Engine (if using GEE for validation)GEE_PROJECT="your-gee-project-id"# Output pathsOUTPUT_DIR="output"VALIDATION_DATA="validation_points.geojson"# Thresholds (tunable)THRESHOLD_NDVI=0.2THRESHOLD_NDWI=0.1THRESHOLD_NDWI_LOW=0.05THRESHOLD_NDWI_HIGH=0.15```**To use this .env file in Python**:```pythonfrom dotenv import load_dotenvimport osload_dotenv('.env')stac_endpoint = os.getenv('STAC_ENDPOINT')output_dir = os.getenv('OUTPUT_DIR')```**Safety Note**:- Never commit `.env` files to version control- Add `.env` to your `.gitignore`- Treat API keys as sensitive data